## 1. Install the openai library

Groq's API is OpenAI-compatible, so we use the openai library with it.
Install it once with pip.

In [1]:
%pip install openai

  Using cached httpx2-2.16.0-py3-none-any.whl.metadata (10 kB)
  Using cached jiter-0.19.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached pydantic-2.14.3-py3-none-any.whl.metadata (141 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached httpcore2-2.16.0-py3-none-any.whl.metadata (27 kB)
  Using cached truststore-0.10.5-py3-none-any.whl.metadata (4.5 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.50.4-cp311-cp311-macosx_11_0_arm64.whl.metadata (6.6 kB)
  Using cached typing_inspection-0.4.5-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 13.3 MB/s  0:00:00 eta 0:00:01
Using cached httpx2-2.16.0-py3-none-any.whl (97 kB)
Using cached httpcore2-2.16.0-py3-none-any.whl (85 kB)
Using cached jiter-0.19.0-cp311-cp311-macosx_11_0_arm64.whl (318 kB)
Using cached pydantic-2.14.3-py3-none-any.whl (494 kB)
Using cached pydantic_core-2.50.4-cp311-cp3

## 2. Connect to Groq

Create the client with your Groq key (typed hidden with getpass) and Groq's base URL.
The same code works with any OpenAI-compatible provider.

In [51]:
import getpass
from openai import OpenAI
client = OpenAI(api_key=getpass.getpass("Groq API key: "), base_url="https://api.groq.com/openai/v1")

Groq API key:  ········


## 3. Choose the model

openai/gpt-oss-120b is the same open model as in the previous session.
It is a reasoning model: it thinks before it answers.

In [52]:
MODEL = "openai/gpt-oss-120b"

## 4. Two helpers: ask() and answer()

ask() sends the messages and returns the whole response. answer() returns only the text.
For this reasoning model, ask() adds 1000 to max_tokens because the thinking uses tokens too. The first test also prints the full response.

In [71]:
def ask(messages, **settings):
    """Send the messages to the chosen model and return the full response."""
    if "gpt-oss" in MODEL and "max_tokens" in settings:
        settings["max_tokens"] += 1000        # a reasoning model thinks first, and the thinking uses tokens too
    return client.chat.completions.create(model=MODEL, messages=messages, **settings)

def answer(response):
    """Just the text of the answer."""
    print (response)
    return response.choices[0].message.content

print(answer(ask([{"role": "user", "content": "Name the three golden signals of monitoring."}])))

ChatCompletion(id='chatcmpl-a6282af5-7483-40be-9b4c-01e72fcf27ce', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The three classic “golden signals” that many teams use to get a quick health snapshot of a service are:\n\n1. **Latency** – How long requests take to complete (often broken out into “good” vs. “bad” latency).  \n2. **Traffic** – The volume of work being handled (requests per second, bytes transferred, etc.).  \n3. **Errors** – The rate of failed requests or error responses (HTTP\u202f5xx, exceptions, time‑outs, etc.).\n\n*(In later SRE practice a fourth signal, **saturation**, is often added, but the original “golden‑signals” triad is latency, traffic, and errors.)*', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='User asks: "Name the three golden signals of monitoring." This is a straightforward question. The golden signals are latency, traffic, errors, and 

## 5. Write a prompt with constraints

The prompt asks for an incident summary and adds rules: exactly 3 bullets, at most 12 words each, only facts from the text.
An f-string puts the incident text into the prompt.

In [72]:
incident = ("At 02:10 UTC the checkout service started returning 502 errors right after the v2.3 deploy. "
            "About 30% of requests are failing.")
prompt = f"""Summarise this incident.

Rules:
- Exactly 3 bullet points
- Maximum 12 words per bullet
- Bullet 1: what happened. Bullet 2: the impact. Bullet 3: the cause
- Use only facts from the text. If the text does not say, write "unknown"

Incident: {incident}"""

## 6. Look at the final prompt

Print the prompt to check that the rules and the incident text are both in it.
This is exactly what the LLM will receive.

In [73]:
prompt

'Summarise this incident.\n\nRules:\n- Exactly 3 bullet points\n- Maximum 12 words per bullet\n- Bullet 1: what happened. Bullet 2: the impact. Bullet 3: the cause\n- Use only facts from the text. If the text does not say, write "unknown"\n\nIncident: At 02:10 UTC the checkout service started returning 502 errors right after the v2.3 deploy. About 30% of requests are failing.'

## 7. Wrap the prompt in a user message

messages is a list. Each message has a role, and role user means the person is speaking.
Every call to the LLM takes a list like this.

In [74]:
messages = [{"role": "user", "content": prompt}]

## 8. Send it to the LLM

temperature=0 keeps the answer steady, and max_tokens limits how much it may generate.
ask() returns the whole response object.

In [75]:
llm_response = ask(messages,temperature=0, max_tokens=50)

## 9. Look at the full response

Besides the answer there is metadata, such as the finish reason and the model's reasoning.
Usually you only want the text.

In [76]:
llm_response

ChatCompletion(id='chatcmpl-b5e5a731-bb2e-4dd6-a009-11b946da2905', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='- Checkout service began returning 502 errors.  \n- Approximately 30% of requests failed.  \n- Cause unknown.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='We need to produce exactly 3 bullet points, each max 12 words. Bullet 1: what happened. Bullet 2: impact. Bullet 3: cause. Use only facts from text; if not said, write "unknown". The incident text: "At 02:10 UTC the checkout service started returning 502 errors right after the v2.3 deploy. About 30% of requests are failing."\n\nBullet 1: what happened: checkout service returned 502 errors. Must be <=12 words. Eg: "Checkout service began returning 502 errors." That\'s 5 words.\n\nBullet 2: impact: 30% of requests failing. Eg: "Approximately 30% of requests failed." That\'s 6 words.\n\nBullet 3: cause: ri

## 10. Read the answer

answer() pulls the text out of the response. The 3 bullets follow the rules.
The cause is unknown because the incident text does not give one.

In [77]:
answer(llm_response)

ChatCompletion(id='chatcmpl-b5e5a731-bb2e-4dd6-a009-11b946da2905', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='- Checkout service began returning 502 errors.  \n- Approximately 30% of requests failed.  \n- Cause unknown.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='We need to produce exactly 3 bullet points, each max 12 words. Bullet 1: what happened. Bullet 2: impact. Bullet 3: cause. Use only facts from text; if not said, write "unknown". The incident text: "At 02:10 UTC the checkout service started returning 502 errors right after the v2.3 deploy. About 30% of requests are failing."\n\nBullet 1: what happened: checkout service returned 502 errors. Must be <=12 words. Eg: "Checkout service began returning 502 errors." That\'s 5 words.\n\nBullet 2: impact: 30% of requests failing. Eg: "Approximately 30% of requests failed." That\'s 6 words.\n\nBullet 3: cause: ri

'- Checkout service began returning 502 errors.  \n- Approximately 30% of requests failed.  \n- Cause unknown.'

## 11. A new question

The user prompt is: A pod keeps restarting. What should I do?
Next we decide who the LLM should be when it answers.

In [78]:
user_prompt = [{"role": "user", "content": "A pod keeps restarting. What should I do?"}]

## 12. Set the LLM's role: the system prompt

The system prompt tells the LLM who it is and how to answer: a senior SRE, at most 3 bullets, each with a kubectl command.
It is the LLM's role, not yours.

In [79]:
system_prompt = "You are a senior SRE. Answer in at most 3 short bullet points, each with a kubectl command."

## 13. Combine the roles into messages

Each message has a role: system for the LLM's role, user for the question.
The next cell is the working version: it builds final_prompt with the question as plain text and sends it.

In [80]:
messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}]

## 14. Same question, a different role

This cell sets a friendly mentor for beginners as the LLM's role and sends the same pod question.
The question is the same, but the answer changes: simple words, at most 4 sentences.

In [84]:
def ask(messages, **settings):
    """Send the messages to the chosen model and return the full response."""
    if "gpt-oss" in MODEL and "max_tokens" in settings:
        settings["max_tokens"] += 1000  # reasoning tokens

    return client.chat.completions.create(
        model=MODEL,
        messages=messages,
        **settings
    )


def answer(response):
    """Just the text of the answer."""
    return response.choices[0].message.content

user_prompt = "A pod keeps restarting. What should I do?"

system_prompt = """You are a friendly mentor for people new to Kubernetes. Explain in simple words, in at most 4 sentences."""

final_prompt = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

llm_responses = ask(
    final_prompt,
    temperature=0,
    max_tokens=50
)

print(answer(llm_responses))

Check the pod’s logs (`kubectl logs <pod>`) to see why it’s crashing, then look at the pod’s events (`kubectl describe pod <pod>`) for clues such as failed health checks or OOM kills. If a liveness/readiness probe is failing, adjust its settings or fix the underlying issue. Make sure the container has enough CPU/memory limits and that required files or environment variables are present. After you correct the problem, delete the pod so the controller can create a fresh, healthy one.


## 15. Add the earlier answer as the assistant role

The LLM has no memory, so its first reply goes back in as an assistant message, followed by the follow-up question.
Without it, 'It says OOMKilled' would mean nothing to the LLM.

In [86]:
final_prompt = [
    {"role": "system", "content": """You are a friendly mentor for people new to Kubernetes. Explain in simple words, in at most 4 sentences."""},
    {"role": "user", "content": "A pod keeps restarting. What should I do?"},
    {"role": "assistant", "content": """Check the pod’s logs (`kubectl logs <pod>`) to see why it’s crashing, then look at the pod’s events (`kubectl describe pod <pod>`) for clues such as failed health checks or OOM kills. If a liveness/readiness probe is failing, adjust its settings or fix the underlying issue. Make sure the container has enough CPU/memory limits and that required files or environment variables are present. After you correct the problem, delete the pod so the controller can create a fresh, healthy one."""},
    {"role": "user", "content": "It says OOMKilled. What next?"},
]

## 16. Send the follow-up

The LLM now knows the context and explains what OOMKilled means.
Every new turn makes the list longer, so the input tokens and the cost grow too.

In [87]:
llm_responses = ask(
    final_prompt,
    temperature=0,
    max_tokens=50
)

print(answer(llm_responses))

`OOMKilled` means the container ran out of memory and the kernel stopped it. Increase the memory request/limit for the pod (e.g., add `resources: { requests: { memory: "256Mi" }, limits: { memory: "512Mi" } }` in the pod spec) or reduce the memory the application uses. After updating the manifest, apply the change (`kubectl apply -f …`) and let the controller recreate the pod. If the app still exceeds the limit, keep raising the limit or profile the code to find memory leaks.


## 17. Private data: a runbook in the prompt

The runbook is your private data, which the LLM never saw in training.
Put it together with the question: What is the production Argo CD URL?

In [91]:
runbook = """
Platform Runbook: Tools and Contacts

Argo CD
- Production: https://argocd.prod.shopfast.example.com
- Staging: https://argocd.stage.shopfast.example.com
- Login with company SSO. You must be on the VPN.

Grafana
- Production dashboards: https://grafana.prod.shopfast.example.com
- Start with the dashboard named "Platform Overview".

On-call
- The primary on-call changes every week. See the PagerDuty schedule "platform-primary".
- If the primary does not answer in 15 minutes, escalate to the SRE lead.

Rollback
- Open the application in Argo CD, choose History and rollback, and pick the last good version.
"""
prompt = f"""
What is the production Argo CD URL?

rules:

runbook: {runbook}"""

## 18. Set the role for the platform agent

The system prompt makes the LLM a platform assistant agent for the operations teams.
The user message carries the question and the runbook.

In [92]:
final_prompt = [
    {"role": "system", "content": """You are the platform assistant agent to help the operations teams"""},
    {"role": "user", "content": prompt}
]

## 19. Ask and read the answer

The LLM finds the production URL inside the runbook you gave it.
A real runbook is hundreds of pages, so this cannot work every time. That is why we need RAG.

In [93]:
llm_responses = ask(
    final_prompt,
    temperature=0,
    max_tokens=50
)

print(answer(llm_responses))

The production Argo CD URL is:

**https://argocd.prod.shopfast.example.com**

*(You’ll need to be on the VPN and log in via the company SSO.)*
